# AeroPulse PM2.5 — Phase 5: Baseline Model Benchmark

Compare persistence, Random Forest, XGBoost, LightGBM and CatBoost on one frozen event-aware dataset.

## Objective

Compare strong tabular baselines on the **same frozen event-aware dataset**.

Models:

1. persistence baseline
2. linear/ElasticNet
3. Random Forest
4. XGBoost
5. LightGBM
6. CatBoost

No model gets a different dataset.

The notebook evaluates:

- overall MAE/RMSE/R²
- pollution-regime errors
- extreme-event recall
- spatial holdout
- temporal holdout
- multi-horizon performance

In [1]:
# Core configuration
from pathlib import Path
import os, json, math, warnings, hashlib, time
import numpy as np
import pandas as pd

from dotenv import find_dotenv, load_dotenv

# Load this folder's .env so os.getenv() below can see the API keys.
_dotenv_path = find_dotenv(usecwd=True)
if not _dotenv_path:
    raise RuntimeError("No .env found. Create one in this folder with OPENAQ_API_KEY / FIRMS_MAP_KEY.")
load_dotenv(_dotenv_path, override=True)
print("Loaded env from:", _dotenv_path)


warnings.filterwarnings("ignore")

PROJECT_ROOT = Path(os.getenv("AEROPULSE_PROJECT_ROOT", ".")).resolve()
DATA_ROOT = PROJECT_ROOT / "data" / "pm25"
RAW_DIR = DATA_ROOT / "raw"
PROCESSED_DIR = DATA_ROOT / "processed"
ARTIFACT_DIR = PROJECT_ROOT / "artifacts" / "pm25"
for p in [RAW_DIR, PROCESSED_DIR, ARTIFACT_DIR]:
    p.mkdir(parents=True, exist_ok=True)

CONFIG = {
    "start": os.getenv("PM25_START", "2023-01-01T00:00:00Z"),
    "end": os.getenv("PM25_END", "2026-01-01T00:00:00Z"),
    "country": "IN",
    "min_station_rows": int(os.getenv("MIN_STATION_ROWS", "3000")),
    "min_coverage": float(os.getenv("MIN_STATION_COVERAGE", "0.35")),
    "india_bbox": {
        "min_lat": 6.0, "max_lat": 37.5,
        "min_lon": 68.0, "max_lon": 97.5,
    },
}
print("Project:", PROJECT_ROOT)
print("Data root:", DATA_ROOT)
print("Configuration:", CONFIG)

Loaded env from: /Users/sunil/Documents/hackathon/AeroPulse/AeroPulse/AeroPulse_ML_Notebooks/pm25_estimator/.env
Project: /Users/sunil/Documents/hackathon/AeroPulse/AeroPulse/AeroPulse_ML_Notebooks/pm25_estimator
Data root: /Users/sunil/Documents/hackathon/AeroPulse/AeroPulse/AeroPulse_ML_Notebooks/pm25_estimator/data/pm25
Configuration: {'start': '2023-01-01T00:00:00Z', 'end': '2026-01-01T00:00:00Z', 'country': 'IN', 'min_station_rows': 3000, 'min_coverage': 0.35, 'india_bbox': {'min_lat': 6.0, 'max_lat': 37.5, 'min_lon': 68.0, 'max_lon': 97.5}}


In [2]:
# ============================================================================
# MODELLING IMPORTS
# ============================================================================
# root_mean_squared_error, not mean_squared_error(squared=False): the `squared`
# argument was deprecated in scikit-learn 1.4 and REMOVED in 1.6, so the old
# call raises TypeError on any current install.
#
# Gradient-boosting libraries are optional, so availability is resolved once
# here rather than inside try/except blocks scattered through the notebook -
# otherwise a later cell references a name that was never bound.
# ============================================================================
import time
from sklearn.metrics import (
    mean_absolute_error, root_mean_squared_error, r2_score,
    precision_score, recall_score, f1_score,
)
from sklearn.impute import SimpleImputer
from sklearn.ensemble import RandomForestRegressor

HAS_XGB = HAS_LGBM = HAS_CAT = False
try:
    from xgboost import XGBRegressor
    HAS_XGB = True
except ImportError:
    pass
try:
    from lightgbm import LGBMRegressor
    HAS_LGBM = True
except ImportError:
    pass
try:
    from catboost import CatBoostRegressor
    HAS_CAT = True
except ImportError:
    pass

print(f"xgboost={HAS_XGB}  lightgbm={HAS_LGBM}  catboost={HAS_CAT}")
if not HAS_LGBM:
    print("WARNING: lightgbm missing - the horizon sweep later in this notebook needs it.")


xgboost=True  lightgbm=True  catboost=True


## 1. Load frozen Phase-2 dataset

Freeze the dataset version before model comparison.

Record the SHA/fingerprint in the model experiment manifest.

In [3]:
EVENT_FILE = PROCESSED_DIR / "event_aware" / "pm25_event_aware_features.parquet"
df = pd.read_parquet(EVENT_FILE)
df["timestamp_utc"] = pd.to_datetime(df["timestamp_utc"], utc=True)

print(df.shape)
print(df["season"].value_counts(dropna=False))
print(df["pollution_regime"].value_counts(dropna=False))

(1705252, 105)
season
monsoon         673073
pre_monsoon     562191
winter          280215
post_monsoon    189773
Name: count, dtype: int64
pollution_regime
normal       912999
moderate     540150
high         163172
very_high     70258
extreme       18673
Name: count, dtype: int64


## 2. Define the baseline feature contract

Exclude:

- current target `pm25` if forecasting from t-1 rather than t
- event labels derived from future/target information
- target columns
- post-event fields
- identifiers that cause memorization unless explicitly tested

Keep current/historical observations, weather, fire, spatial, seasonal and regional features.

For the first baseline experiment, predict `t+1h`.

In [4]:
# ============================================================================
# TARGET AND FEATURE SELECTION
# ============================================================================
# What must be excluded, and why:
#   target_*     the answers, at every horizon
#   event_*      computed from the FULL event including its future, so they
#                leak the outcome backwards into rows before the peak
#   identifiers  station_name / source / unit carry no signal
#
# What is deliberately INCLUDED, and why:
#   pm25         the CURRENT observation. This is known at prediction time, so
#                using it to forecast t+h is not leakage - it is the single
#                most informative feature available, and it is what the
#                persistence baseline uses. The earlier version excluded it
#                while keeping pm25_lag_1h, which handicapped every model
#                against a baseline that effectively had fresher information.
# ============================================================================
TARGET_H = int(os.getenv("PM25_TARGET_H", "1"))
TARGET = f"target_pm25_t_plus_{TARGET_H}h"
if TARGET not in df.columns:
    raise ValueError(f"{TARGET} not in dataset. Available: "
                     f"{[c for c in df.columns if c.startswith('target_pm25')]}")

EXCLUDE_PREFIXES = ("target_", "event_")
EXCLUDE = {
    "timestamp_utc", "timestamp_local", "station_name", "source", "pm25_unit",
    "pollution_regime", "country", "state", "city", "source_quality",
    "schema_version", "sensor_id", "grid_id", "weather_source", "fire_source",
}

FEATURES = [
    c for c in df.columns
    if c not in EXCLUDE
    and not c.startswith(EXCLUDE_PREFIXES)
    and pd.api.types.is_numeric_dtype(df[c])
]

# Guard: any feature perfectly correlated with the target is leakage.
_probe = df[FEATURES + [TARGET]].dropna().head(20000)
if len(_probe) > 100:
    corr = _probe[FEATURES].corrwith(_probe[TARGET]).abs().sort_values(ascending=False)
    suspicious = corr[corr > 0.999]
    print("Top feature/target correlations:")
    print(corr.head(8).to_string())
    if len(suspicious):
        print(f"\nLEAKAGE WARNING - near-perfect correlation: {list(suspicious.index)}")

print(f"\nTarget       : {TARGET} (horizon {TARGET_H}h)")
print(f"Feature count: {len(FEATURES)}")


Top feature/target correlations:
pm25                   0.924959
pm25_lag_1h            0.851659
pm25_roll_mean_3h      0.827630
neighbour_pm25_mean    0.814506
pm25_roll_mean_6h      0.790606
pm25_roll_mean_24h     0.770305
pm25_lag_3h            0.753819
neighbour_pm25_max     0.747774

Target       : target_pm25_t_plus_1h (horizon 1h)
Feature count: 75


## 3. Leakage-safe temporal split

Use:

- train: earliest 70%
- validation: next 15%
- test: final 15%

Do not shuffle.

Then separately create a spatial holdout and an event/season holdout.

The final report must never present a random split as the production metric.

In [5]:
# ============================================================================
# CHRONOLOGICAL SPLIT - train / validation / test
# ============================================================================
# Never split randomly here. Rows one hour apart are almost identical, so a
# random split puts near-duplicates of test rows into training and inflates
# every score. Splitting on time reproduces the real task: predict forward.
#
# The validation slice sits BETWEEN train and test so it can be used for early
# stopping and probability calibration without ever touching the test period.
# ============================================================================
df = df.dropna(subset=[TARGET]).sort_values("timestamp_utc")
t1 = df.timestamp_utc.quantile(0.70)
t2 = df.timestamp_utc.quantile(0.85)

train = df[df.timestamp_utc < t1].copy()
valid = df[(df.timestamp_utc >= t1) & (df.timestamp_utc < t2)].copy()
test = df[df.timestamp_utc >= t2].copy()

print(f"train {train.shape}  valid {valid.shape}  test {test.shape}")
print(f"train: {train.timestamp_utc.min()} -> {train.timestamp_utc.max()}")
print(f"valid: {valid.timestamp_utc.min()} -> {valid.timestamp_utc.max()}")
print(f"test : {test.timestamp_utc.min()} -> {test.timestamp_utc.max()}")
assert train.timestamp_utc.max() <= valid.timestamp_utc.min(), "train/valid overlap"
assert valid.timestamp_utc.max() <= test.timestamp_utc.min(), "valid/test overlap"

# Regime mix in the test period decides whether extreme-event metrics mean
# anything: a handful of extreme rows gives a very noisy recall figure.
print("\nTest-period regime mix:")
print(pd.cut(test[TARGET], [-np.inf,30,60,90,150,np.inf],
             labels=["normal","moderate","high","very_high","extreme"]
             ).value_counts().sort_index().to_string())


train (1193484, 105)  valid (255820, 105)  test (255799, 105)
train: 2025-02-18 20:30:00+00:00 -> 2026-03-21 23:30:00+00:00
valid: 2026-03-22 00:30:00+00:00 -> 2026-06-13 15:30:00+00:00
test : 2026-06-13 16:30:00+00:00 -> 2026-09-09 06:30:00+00:00

Test-period regime mix:
target_pm25_t_plus_1h
normal       206703
moderate      43610
high           3648
very_high      1252
extreme         586


## 4. Evaluation helpers

Extreme-event performance is a first-class metric.

A model that has excellent average MAE but misses every extreme event is not acceptable for AeroPulse alerting.

In [6]:
# ============================================================================
# EVALUATION METRICS
# ============================================================================
# Three views, because a single headline number hides the failure that matters:
#   regression_metrics  overall accuracy
#   regime_metrics      accuracy BY pollution level - the model can look good
#                       overall while being useless during severe episodes
#   event_metrics       treats forecasting as alerting: did we fire when we
#                       should have? Recall is the number that matters for a
#                       public-health warning; a missed episode is far more
#                       costly than a false alarm.
# ============================================================================
def regression_metrics(y, p):
    return {
        "MAE": mean_absolute_error(y, p),
        "RMSE": root_mean_squared_error(y, p),
        "R2": r2_score(y, p),
    }


def regime_metrics(y, p):
    y, p = np.asarray(y), np.asarray(p)
    bins = [-np.inf, 30, 60, 90, 150, np.inf]
    labels = ["normal", "moderate", "high", "very_high", "extreme"]
    regime = pd.cut(y, bins=bins, labels=labels)
    rows = []
    for r in labels:
        mask = np.asarray(regime == r)
        if mask.sum():
            rows.append({
                "regime": r, "count": int(mask.sum()),
                "MAE": mean_absolute_error(y[mask], p[mask]),
                "RMSE": root_mean_squared_error(y[mask], p[mask]),
                "bias": float(np.mean(p[mask] - y[mask])),
            })
    return pd.DataFrame(rows)


def event_metrics(y, p, threshold=150):
    yt = np.asarray(y) >= threshold
    yp = np.asarray(p) >= threshold
    return {
        "threshold": threshold,
        "precision": precision_score(yt, yp, zero_division=0),
        "recall": recall_score(yt, yp, zero_division=0),
        "f1": f1_score(yt, yp, zero_division=0),
        "actual_events": int(yt.sum()),
        "predicted_events": int(yp.sum()),
    }


print("Metric helpers ready (regression / per-regime / event-alerting).")


Metric helpers ready (regression / per-regime / event-alerting).


### Is a single chronological split enough?

The next three cells examine the split strategy itself, because it determines
whether every number in this notebook means anything.

1. **Season coverage** — what the test block actually contains.
2. **Random-split leakage** — a measurement of why we do *not* shuffle.
3. **Rolling-origin CV** — the correct way to get seasonal coverage.


In [7]:
# ============================================================================
# WHAT SEASON DOES THE TEST SET ACTUALLY COVER?
# ============================================================================
# The weakness of a single chronological split: the test block is whatever
# season fell last. If the model is only ever scored on monsoon air, its
# burning-season behaviour is untested no matter how good the headline looks.
# ============================================================================
print("Season mix per split (rows):")
mix = pd.DataFrame({
    "train": train.season.value_counts(),
    "valid": valid.season.value_counts(),
    "test":  test.season.value_counts(),
}).fillna(0).astype(int)
display(mix)

print("\nExtreme rows (>=150 ug/m3) per split:")
for name, part in [("train", train), ("valid", valid), ("test", test)]:
    n = int((part[TARGET] >= 150).sum())
    print(f"  {name:6} {n:6,}  ({n/max(len(part),1):.3%} of the split)")

missing_in_test = set(train.season.dropna()) - set(test.season.dropna())
if missing_in_test:
    print(f"\nWARNING: seasons present in TRAIN but absent from TEST: "
          f"{sorted(missing_in_test)}")
    print("The headline metrics say nothing about those regimes. The "
          "rolling-origin CV below is what covers them.")


Season mix per split (rows):


,train,valid,test
season,,,
monsoon,377010,40115,255799
post_monsoon,189773,0,0
pre_monsoon,346486,215705,0
winter,280215,0,0



Extreme rows (>=150 ug/m3) per split:
  train  17,242  (1.445% of the split)
  valid   1,222  (0.478% of the split)
  test      592  (0.231% of the split)



The headline metrics say nothing about those regimes. The rolling-origin CV below is what covers them.


### Result — ⚠️ the test set is 100% monsoon

| Season | train | valid | **test** |
|---|---:|---:|---:|
| monsoon | 377,010 | 40,115 | **255,799** |
| pre_monsoon | 346,486 | 215,705 | **0** |
| winter | 280,215 | 0 | **0** |
| post_monsoon | 189,773 | 0 | **0** |

Extreme rows: train **1.445%** → valid 0.478% → test **0.231%**.

**This confirms the concern about splitting by time.** Three of four regimes — including both dirty seasons — never appear in the test set at all. Every headline number below describes the cleanest air of the year, and says nothing about the burning season or winter inversions.

The fix is *more temporal folds*, not random ones. See the next two cells.


### Why not just shuffle the rows?

Mixing all months and splitting randomly is tempting — it would give every
split a balanced season mix. It also destroys the evaluation. The next cell
trains the **same model on the same data** under both regimes so the size of
the leak is measured rather than argued about.

See `MODEL_IMPROVEMENTS.md` §1 for the full reasoning.


In [8]:
# ============================================================================
# WHY NOT A RANDOM SPLIT?  (run this once; it is a demonstration, not a step)
# ============================================================================
# A random shuffle-split looks attractive here: it mixes all seasons into both
# train and test, so the test set is no longer "whatever season happened to
# fall last". That concern is legitimate - but a random split is the wrong
# cure, and this cell measures exactly how wrong.
#
# THE LEAK, concretely. Rows are hourly and consecutive:
#
#   row t-1 :  pm25 = 180                     <- lands in TRAIN
#   row t   :  pm25_lag_1h = 180, target = ?  <- lands in TEST
#   row t+1 :  pm25 = 195                     <- lands in TRAIN
#              ...and target_pm25_t_plus_1h for row t IS 195.
#
# So for a test row, the answer is sitting in the training set as an ordinary
# feature of its neighbour. With hourly data and an 85/15 split, ~85% of every
# test row's neighbours are in train. The model does not have to learn
# atmospheric physics - it can interpolate between two rows it memorised.
#
# The numbers below are produced by training the SAME model on the SAME data
# under both regimes. Treat the random-split score as a measurement of the
# leak, not as accuracy.
# ============================================================================
from sklearn.model_selection import train_test_split

_probe_feats = [c for c in FEATURES]
_d = df.dropna(subset=[TARGET])

def _fit_score(tr, te, label):
    imp = SimpleImputer(strategy="median")
    m = LGBMRegressor(n_estimators=400, learning_rate=0.05, num_leaves=63,
                      random_state=42, verbose=-1)
    m.fit(imp.fit_transform(tr[_probe_feats]), tr[TARGET])
    p = m.predict(imp.transform(te[_probe_feats]))
    s = regression_metrics(te[TARGET], p)
    print(f"  {label:34} MAE {s['MAE']:6.2f}   RMSE {s['RMSE']:6.2f}   R2 {s['R2']:.4f}")
    return s

print("Same model, same data, two split strategies:\n")
rnd_tr, rnd_te = train_test_split(_d, test_size=0.15, random_state=42, shuffle=True)
rnd = _fit_score(rnd_tr, rnd_te, "RANDOM split (LEAKY)")

_cut = _d.timestamp_utc.quantile(0.85)
chron = _fit_score(_d[_d.timestamp_utc < _cut], _d[_d.timestamp_utc >= _cut],
                   "CHRONOLOGICAL split (honest)")

inflation = rnd["R2"] - chron["R2"]
print(f"\n  R2 inflation from leakage: {inflation:+.4f}")
print(f"  RMSE understated by      : {chron['RMSE'] - rnd['RMSE']:.2f} ug/m3 "
      f"({(chron['RMSE']-rnd['RMSE'])/chron['RMSE']:.0%})")
print("\nThe random-split number is what the model scores when it can look up "
      "its neighbours. The chronological number is what it scores when asked "
      "to do the actual job: predict a time it has never seen. Only the second "
      "one predicts production behaviour.")


Same model, same data, two split strategies:



  RANDOM split (LEAKY)               MAE   5.57   RMSE  16.86   R2 0.7646


  CHRONOLOGICAL split (honest)       MAE   4.39   RMSE  14.04   R2 0.5224

  R2 inflation from leakage: +0.2422
  RMSE understated by      : -2.82 ug/m3 (-20%)

The random-split number is what the model scores when it can look up its neighbours. The chronological number is what it scores when asked to do the actual job: predict a time it has never seen. Only the second one predicts production behaviour.


### Result — random splitting inflates R² by +0.24

| Split | MAE | RMSE | R² |
|---|---:|---:|---:|
| **Random (leaky)** | 5.57 | 16.86 | **0.7646** |
| **Chronological (honest)** | 4.39 | 14.04 | **0.5224** |

Same model, same features, same data. The only difference is how rows were assigned.

**The shuffled model looks 46% better purely by reading its neighbours.** With hourly rows and an 85/15 shuffle, ~85% of every test row's adjacent hours land in training — and `target_pm25_t_plus_1h` for row *t* is literally the `pm25` feature of row *t+1*. The model interpolates rather than forecasts.

Shipping on the random-split number would mean promising R² 0.76 and delivering 0.52 in production. The 0.24 gap is a measurement of the leak, not of skill.


### Rolling-origin cross-validation

The right fix for "the test set is only one season" is **more temporal
splits**, not random ones. Each fold trains on the past and tests on the block
after it, with a purge + embargo gap so `shift(-h)` targets and 24 h rolling
windows cannot straddle the boundary.

Read the **spread across folds**, not just the mean.


In [9]:
# ============================================================================
# ROLLING-ORIGIN CROSS-VALIDATION  (the right fix for "test is only one season")
# ============================================================================
# The concern behind "why not shuffle?" is real: a single chronological split
# tests on whatever season happens to fall last, so a model can be scored
# entirely on monsoon air and never evaluated on the burning season.
#
# The fix is MORE temporal splits, not random ones. Each fold trains on
# everything up to a cut and tests on the block after it, so every fold lands
# in a different part of the year while causality is preserved:
#
#   fold 1  train |=====|  test |--|
#   fold 2  train |========|     test |--|
#   fold 3  train |===========|        test |--|
#
# PURGE / EMBARGO. The target at time t is pm25 at t+h, so rows in the last h
# hours of a training block describe times that fall inside the test block.
# Those rows are dropped ("purged"); the embargo drops a further margin so
# rolling windows spanning the boundary cannot leak either. Without this the
# split is still chronological but not clean.
# ============================================================================
def rolling_origin_folds(frame, n_splits=5, test_frac=0.12,
                         purge_hours=None, embargo_hours=48):
    """Yield (train_idx, test_idx) for expanding-window folds, purged."""
    purge_hours = purge_hours if purge_hours is not None else TARGET_H
    f = frame.sort_values("timestamp_utc")
    t_min, t_max = f.timestamp_utc.min(), f.timestamp_utc.max()
    span = t_max - t_min
    test_len = span * test_frac
    # Place the folds so the last one ends at the end of the series.
    first_cut = t_min + span * (1 - test_frac * n_splits)
    gap = pd.Timedelta(hours=purge_hours + embargo_hours)

    for k in range(n_splits):
        cut = first_cut + test_len * k
        test_end = cut + test_len
        tr = f[f.timestamp_utc < cut - gap]
        te = f[(f.timestamp_utc >= cut) & (f.timestamp_utc < test_end)]
        if len(tr) < 500 or len(te) < 100:
            continue
        yield tr, te, cut, test_end


def evaluate_rolling(frame, model_factory, label, n_splits=5):
    rows = []
    for i, (tr, te, cut, end) in enumerate(rolling_origin_folds(frame, n_splits), 1):
        imp = SimpleImputer(strategy="median")
        m = model_factory()
        m.fit(imp.fit_transform(tr[FEATURES]), tr[TARGET])
        p = m.predict(imp.transform(te[FEATURES]))
        seasons = te["season"].value_counts()
        rows.append({
            "fold": i,
            "test_start": cut.strftime("%Y-%m-%d"),
            "test_end": end.strftime("%Y-%m-%d"),
            "dominant_season": seasons.index[0] if len(seasons) else "n/a",
            "train_rows": len(tr), "test_rows": len(te),
            **regression_metrics(te[TARGET], p),
            "extreme_rows": int((te[TARGET] >= 150).sum()),
            "extreme_recall": event_metrics(te[TARGET], p)["recall"],
        })
    out = pd.DataFrame(rows)
    if not out.empty:
        print(f"\n{label} - rolling-origin CV")
        display(out)
        print(f"  mean R2 {out.R2.mean():.4f}  (sd {out.R2.std():.4f})   "
              f"mean RMSE {out.RMSE.mean():.2f}")
        print("  The SPREAD across folds is the useful number: a model whose R2 "
              "swings widely between seasons is not one season away from being "
              "reliable, it is unreliable.")
    return out


if HAS_LGBM:
    cv_lgbm = evaluate_rolling(
        df, lambda: LGBMRegressor(n_estimators=600, learning_rate=0.05,
                                  num_leaves=63, random_state=42, verbose=-1),
        "LightGBM", n_splits=5)
else:
    cv_lgbm = pd.DataFrame()
    print("lightgbm unavailable - rolling CV skipped.")



LightGBM - rolling-origin CV


,fold,test_start,test_end,dominant_season,train_rows,test_rows,MAE,RMSE,R2,extreme_rows,extreme_recall
0,1,2025-10-03,2025-12-10,post_monsoon,693375,213713,8.097730,20.390557,0.797735,6427,0.706084
1,2,2025-12-10,2026-02-17,winter,906473,179911,7.430597,16.614831,0.847450,6113,0.727793
2,3,2026-02-17,2026-04-26,pre_monsoon,1089132,208906,5.876852,14.725331,0.739825,1515,0.504290
3,4,2026-04-26,2026-07-03,pre_monsoon,1295655,200180,5.308756,20.232863,0.456261,767,0.315515
4,5,2026-07-03,2026-09-09,monsoon,1495925,202955,4.287064,14.110455,0.509400,473,0.393235


  mean R2 0.6701  (sd 0.1762)   mean RMSE 17.21
  The SPREAD across folds is the useful number: a model whose R2 swings widely between seasons is not one season away from being reliable, it is unreliable.


### Result — and the conclusion reverses

| Fold | Test period | Season | R² | RMSE | Extreme recall |
|---|---|---|---:|---:|---:|
| 1 | Oct–Dec 2025 | post_monsoon | **0.798** | 20.39 | **0.706** |
| 2 | Dec–Feb 2026 | winter | **0.847** | 16.61 | **0.728** |
| 3 | Feb–Apr 2026 | pre_monsoon | 0.740 | 14.73 | 0.504 |
| 4 | Apr–Jul 2026 | pre_monsoon | 0.456 | 20.23 | 0.316 |
| 5 | Jul–Sep 2026 | monsoon | 0.509 | 14.11 | 0.393 |
| | | **mean** | **0.670** (sd 0.176) | 17.21 | |

**This is the most important table in the notebook, and it overturns the headline.**

The single chronological split reported R² 0.52 — but it only ever tested on monsoon, which turns out to be the model's *worst* regime. Across folds the model reaches **R² 0.85 in winter and 0.80 in post-monsoon, with extreme recall above 0.70** in both.

Why the reversal: monsoon PM2.5 is low and its variance is driven by rainfall washout, which is close to noise at hourly resolution. Winter and post-monsoon variance is driven by boundary-layer collapse and fire — both *observable in the features*. The model is strongest exactly where the health risk is highest.

The fold spread (sd 0.176) is itself the reliability measure. A model swinging 0.46 → 0.85 across the year needs season-aware confidence reporting, not a single advertised accuracy.


### Leave-one-season-out diagnostic

A different question from the one above: *has the model learned each regime at
all?* Holding out one season entirely and training on the rest is not a
production estimate — it trains on future data to predict the past — but it
isolates whether a regime is representable, which is what a debugging pass
needs to know.


In [10]:
# ============================================================================
# LEAVE-ONE-SEASON-OUT  (diagnostic only, NOT a production estimate)
# ============================================================================
# This trains on future data to predict the past, so it does not estimate
# deployed accuracy. It answers a narrower question: given everything else the
# model has seen, can it represent THIS regime? A season that scores badly here
# is one the feature set cannot describe - a modelling gap, not a split artefact.
# ============================================================================
if HAS_LGBM:
    rows = []
    for season in df.season.dropna().unique():
        tr = df[df.season != season].dropna(subset=[TARGET])
        te = df[df.season == season].dropna(subset=[TARGET])
        if len(te) < 200 or len(tr) < 500:
            continue
        imp = SimpleImputer(strategy="median")
        m = LGBMRegressor(n_estimators=600, learning_rate=0.05, num_leaves=63,
                          random_state=42, verbose=-1)
        m.fit(imp.fit_transform(tr[FEATURES]), tr[TARGET])
        p = m.predict(imp.transform(te[FEATURES]))
        rows.append({"held_out_season": season, "rows": len(te),
                     **regression_metrics(te[TARGET], p),
                     "extreme_rows": int((te[TARGET] >= 150).sum()),
                     "extreme_recall": event_metrics(te[TARGET], p)["recall"],
                     "mean_pm25": float(te[TARGET].mean())})
    season_cv = pd.DataFrame(rows).sort_values("R2")
    display(season_cv)
    if not season_cv.empty:
        worst = season_cv.iloc[0]
        print(f"Weakest regime: {worst.held_out_season} "
              f"(R2 {worst.R2:.3f}, mean PM2.5 {worst.mean_pm25:.0f} ug/m3)")
        print("If the weakest season is post_monsoon or winter, the model is "
              "worst exactly where the health risk is highest.")
else:
    season_cv = pd.DataFrame()


,held_out_season,rows,MAE,RMSE,R2,extreme_rows,extreme_recall,mean_pm25
2,monsoon,672924,4.532575,15.564108,0.523554,1915,0.412010,22.750472
1,pre_monsoon,562191,5.934717,19.042130,0.629538,4047,0.442303,36.882282
3,post_monsoon,189773,6.985558,19.300989,0.816805,5102,0.739906,51.821544
0,winter,280215,7.158459,16.361204,0.839208,7992,0.722598,58.724110


Weakest regime: monsoon (R2 0.524, mean PM2.5 23 ug/m3)
If the weakest season is post_monsoon or winter, the model is worst exactly where the health risk is highest.


### Result — confirms the same ordering

| Held-out season | Rows | R² | Extreme recall | Mean PM2.5 |
|---|---:|---:|---:|---:|
| winter | 280,215 | **0.839** | **0.723** | 58.7 |
| post_monsoon | 189,773 | **0.817** | **0.740** | 51.8 |
| pre_monsoon | 562,191 | 0.630 | 0.442 | 36.9 |
| **monsoon** | 672,924 | **0.524** | 0.412 | 22.8 |

Independent of the rolling-CV design, the same ranking appears: the dirty seasons are the *easy* ones to model, and monsoon is the hardest.

This is a diagnostic, not a production estimate — it trains on future data to predict the past. Its purpose is narrower: to show that no regime is unrepresentable by the current feature set. None scores badly enough to indicate a missing physical driver.


## 5. Persistence baseline

In [11]:
# ============================================================================
# PERSISTENCE BASELINE - "tomorrow looks like today"
# ============================================================================
# Every model below must beat this or it is not earning its complexity.
#
# FIXED: the baseline previously used pm25_lag_1h, i.e. the value from t-1,
# to predict t+1 - a two-hour-old reading. That is not persistence, it is a
# handicapped version of it, and it made every model look better than it was.
# Correct persistence for horizon h is the CURRENT observation, pm25(t).
# ============================================================================
y_test = test[TARGET].to_numpy()
persistence = test["pm25"].to_numpy()

print(f"Persistence baseline (pm25(t) -> t+{TARGET_H}h):")
print(" ", regression_metrics(y_test, persistence))
print(" ", event_metrics(y_test, persistence))
print("\nFor reference, the previous (incorrect) lag-1h baseline:")
print(" ", regression_metrics(y_test, test["pm25_lag_1h"].fillna(test["pm25"]).to_numpy()))


Persistence baseline (pm25(t) -> t+1h):
  {'MAE': 4.535613755331334, 'RMSE': 16.00386431543627, 'R2': 0.3795335470410123}
  {'threshold': 150, 'precision': 0.4814189189189189, 'recall': 0.4814189189189189, 'f1': 0.4814189189189189, 'actual_events': 592, 'predicted_events': 592}

For reference, the previous (incorrect) lag-1h baseline:
  {'MAE': 6.414220868728963, 'RMSE': 19.719042030008747, 'R2': 0.058023194317715854}


### Result — and why the old baseline was flattering everything

| Baseline | MAE | RMSE | R² | Extreme recall |
|---|---:|---:|---:|---:|
| **`pm25(t)` — correct** | **4.54** | **16.00** | **0.380** | **0.481** |
| `pm25_lag_1h` — previous, incorrect | 6.41 | 19.72 | 0.058 | — |

The previous code used `pm25_lag_1h` — the reading from *t−1* — to predict *t+1*, a two-hour-old value. That is not persistence; it is a handicapped version of it, and it made every model look better than it was by a full 1.9 µg/m³ of MAE.

With the correct baseline, persistence achieves **R² 0.380 and extreme recall 0.481**. Hold that recall figure in mind for the comparison table below.


## 6. Prepare matrices

In [12]:
# ============================================================================
# DESIGN MATRICES
# ============================================================================
# The imputer is FIT ON TRAIN ONLY and merely applied to valid/test, so test
# statistics never leak backwards into training.
#
# Tree ensembles from xgboost/lightgbm/catboost handle NaN natively and are
# given the RAW frames below - imputing first would discard the informative
# "this reading is missing" signal. RandomForest cannot, so it gets the
# imputed matrices. That difference is intentional, not an inconsistency.
# ============================================================================
X_train, y_train = train[FEATURES], train[TARGET]
X_valid, y_valid = valid[FEATURES], valid[TARGET]
X_test, y_test_s = test[FEATURES], test[TARGET]
y_test = y_test_s.to_numpy()

imputer = SimpleImputer(strategy="median")
Xtr = imputer.fit_transform(X_train)
Xva = imputer.transform(X_valid)
Xte = imputer.transform(X_test)

print("imputed:", Xtr.shape, Xva.shape, Xte.shape)
print(f"missing-value rate in train features: {X_train.isna().to_numpy().mean():.2%}")


imputed: (1193484, 75) (255820, 75) (255799, 75)
missing-value rate in train features: 0.07%


## 7. Random Forest

In [13]:
# ============================================================================
# RANDOM FOREST baseline
# ============================================================================
# Bagged trees, no boosting - a useful "does gradient boosting actually help?"
# reference point.
#
# max_samples caps each tree at a fraction of the rows. On 1.2M training rows
# an uncapped 400-tree forest needs hours and tens of GB; bagging 25% per tree
# costs very little accuracy because the trees are decorrelated anyway, and it
# is the standard way to run RF at this scale.
# ============================================================================
rf = RandomForestRegressor(
    n_estimators=int(os.getenv("PM25_RF_TREES", "200")),
    min_samples_leaf=5,
    max_features="sqrt",
    max_samples=float(os.getenv("PM25_RF_MAX_SAMPLES", "0.25")),
    n_jobs=-1,
    random_state=42,
)
_t = time.time()
rf.fit(Xtr, y_train)
pred_rf = rf.predict(Xte)
print(f"RandomForest : {regression_metrics(y_test, pred_rf)}  [{time.time()-_t:.0f}s]")


RandomForest : {'MAE': 4.36652270396954, 'RMSE': 13.751064080334512, 'R2': 0.5419201423371747}  [51s]


## 8. XGBoost

In [14]:
# XGBoost. Availability was resolved once in the imports cell, so this cannot
# leave `pred_xgb` undefined for a later cell that assumes the import worked.
if HAS_XGB:
    xgb = XGBRegressor(
        n_estimators=800, max_depth=8, learning_rate=0.03,
        subsample=0.85, colsample_bytree=0.85,
        objective="reg:squarederror", random_state=42, n_jobs=-1,
        early_stopping_rounds=50, eval_metric="rmse",
    )
    xgb.fit(X_train, y_train, eval_set=[(X_valid, y_valid)], verbose=False)
    pred_xgb = xgb.predict(X_test)
    print("XGBoost      :", regression_metrics(y_test, pred_xgb))
    print("  best iteration:", xgb.best_iteration)
else:
    print("xgboost not installed - skipped.")


XGBoost      : {'MAE': 4.559588762260522, 'RMSE': 14.230531634877677, 'R2': 0.509418875794893}
  best iteration: 128


## 9. LightGBM

In [15]:
# LightGBM, with early stopping on the validation slice.
if HAS_LGBM:
    import lightgbm as lgb
    lgbm = LGBMRegressor(
        n_estimators=2000, learning_rate=0.03, num_leaves=63,
        subsample=0.85, colsample_bytree=0.85,
        random_state=42, objective="regression", verbose=-1,
    )
    lgbm.fit(X_train, y_train, eval_set=[(X_valid, y_valid)],
             callbacks=[lgb.early_stopping(50, verbose=False)])
    pred_lgbm = lgbm.predict(X_test)
    print("LightGBM     :", regression_metrics(y_test, pred_lgbm))
    print("  best iteration:", lgbm.best_iteration_)
else:
    print("lightgbm not installed - skipped.")


LightGBM     : {'MAE': 4.397512093099862, 'RMSE': 13.96199959269318, 'R2': 0.5277588520374241}
  best iteration: 307


## 10. CatBoost

In [16]:
if HAS_CAT:
    cat = CatBoostRegressor(iterations=1200, depth=8, learning_rate=0.03,
                            loss_function="RMSE", random_seed=42, verbose=False)
    cat.fit(X_train, y_train, eval_set=(X_valid, y_valid), early_stopping_rounds=50)
    pred_cat = cat.predict(X_test)
    print("CatBoost     :", regression_metrics(y_test, pred_cat))
else:
    print("catboost not installed - skipped.")


CatBoost     : {'MAE': 4.410593410980308, 'RMSE': 13.750537438993609, 'R2': 0.541955228955849}


## 11. Model comparison table

The winner is **not automatically the model with the lowest MAE**.

Use a weighted decision:

- overall RMSE/MAE
- high/very-high MAE
- extreme-event recall
- temporal robustness
- spatial robustness
- inference latency
- model complexity

In [17]:
# ============================================================================
# MODEL COMPARISON
# ============================================================================
# Ranked by extreme-event RECALL first, then RMSE. That ordering is deliberate:
# for a public-health alerting product a missed hazardous episode costs far
# more than a false alarm, and a model with the best RMSE can still have zero
# recall on the events the system exists to warn about.
# ============================================================================
predictions = {"persistence": persistence}
for name, var in [("random_forest", "pred_rf"), ("xgboost", "pred_xgb"),
                  ("lightgbm", "pred_lgbm"), ("catboost", "pred_cat")]:
    if var in globals():
        predictions[name] = np.asarray(globals()[var])

rows = []
for name, pred in predictions.items():
    m = regression_metrics(y_test, pred)
    e = event_metrics(y_test, pred)
    rows.append({"model": name, **m,
                 "extreme_recall": e["recall"], "extreme_precision": e["precision"],
                 "extreme_f1": e["f1"], "predicted_events": e["predicted_events"]})

comparison = pd.DataFrame(rows).sort_values(["extreme_recall", "RMSE"], ascending=[False, True])
display(comparison)

# The single most useful line in this table is whether PERSISTENCE wins.
# If a model cannot beat "the next hour looks like this one", it is not adding
# value at this horizon regardless of how good its RMSE looks.
ml_only = comparison[comparison.model != "persistence"]
best_ml_name = ml_only.iloc[0]["model"] if len(ml_only) else None
persist_row = comparison[comparison.model == "persistence"].iloc[0]
print(f"\nBest ML model      : {best_ml_name}")
print(f"Persistence recall : {persist_row.extreme_recall:.3f} | "
      f"best ML recall: {ml_only.iloc[0].extreme_recall:.3f}")
if persist_row.extreme_recall >= ml_only.extreme_recall.max():
    print("PERSISTENCE BEATS EVERY MODEL ON EXTREME RECALL. The models win on "
          "RMSE by regressing toward the mean, which is exactly the behaviour "
          "that loses hazardous episodes. See MODEL_IMPROVEMENTS.md 2.5-2.7.")

n_extreme = int((y_test >= 150).sum())
print(f"\nActual extreme rows in test: {n_extreme}")
if n_extreme < 30:
    print("CAUTION: too few extreme rows for a stable recall estimate. Treat the "
          "extreme_* columns as indicative only, and widen the dataset window "
          "to include a burning season before drawing conclusions.")


,model,MAE,RMSE,R2,extreme_recall,extreme_precision,extreme_f1,predicted_events
0,persistence,4.535614,16.003864,0.379534,0.481419,0.481419,0.481419,592
4,catboost,4.410593,13.750537,0.541955,0.396959,0.532880,0.454985,441
3,lightgbm,4.397512,13.962000,0.527759,0.386824,0.505519,0.438278,453
2,xgboost,4.559589,14.230532,0.509419,0.385135,0.424581,0.403897,537
1,random_forest,4.366523,13.751064,0.541920,0.376689,0.534772,0.442022,417



Best ML model      : catboost
Persistence recall : 0.481 | best ML recall: 0.397
PERSISTENCE BEATS EVERY MODEL ON EXTREME RECALL. The models win on RMSE by regressing toward the mean, which is exactly the behaviour that loses hazardous episodes. See MODEL_IMPROVEMENTS.md 2.5-2.7.

Actual extreme rows in test: 592


### Result — ⚠️ persistence beats every model on extreme recall

| Model | MAE | RMSE | R² | **Extreme recall** | Extreme precision |
|---|---:|---:|---:|---:|---:|
| **persistence** | 4.54 | 16.00 | 0.380 | **0.481** | 0.481 |
| catboost | 4.41 | **13.75** | **0.542** | 0.397 | 0.533 |
| lightgbm | 4.40 | 13.96 | 0.528 | 0.387 | 0.506 |
| xgboost | 4.56 | 14.23 | 0.509 | 0.385 | 0.425 |
| random_forest | **4.37** | 13.75 | 0.542 | 0.377 | 0.535 |

*(592 actual extreme rows in test)*

**Every ML model beats persistence on RMSE and loses to it on extreme recall.** That is not a contradiction — it is the same fact stated twice. Squared-error training makes a model hedge toward the conditional mean, which lowers RMSE across 250k ordinary rows while systematically *under-shooting* the 592 rows that matter. The models predict 417–537 extreme events where persistence predicts 592.

For a public-health alerting product this ordering matters more than the RMSE column: **the best-RMSE model misses ~60% of hazardous episodes; persistence misses ~52%.**

The three boosted models are within 0.03 R² of each other — they are drawing the same information from the same features, so ensembling them will yield little. The gain has to come from the loss function and the features, not the algorithm. See `MODEL_IMPROVEMENTS.md` §2.5–2.7.


## 12. Regime diagnostics

Always save a regime error table.

The current notebook demonstrated why: a model can look acceptable overall while being unusable for extreme events.

In [18]:
# Per-regime error for the top-ranked model. This is the table that reveals
# whether headline accuracy is carried entirely by ordinary days.
best_name = comparison.iloc[0]["model"]
best_pred = predictions[best_name]
print("Best by extreme recall then RMSE:", best_name)
display(regime_metrics(y_test, best_pred))
print("A monotonically rising MAE across regimes means the model degrades "
      "exactly where accuracy matters most; a large positive bias in the "
      "extreme row means it systematically under-predicts hazardous air.")


Best by extreme recall then RMSE: persistence


,regime,count,MAE,RMSE,bias
0,normal,206703,3.294408,9.854938,0.798526
1,moderate,43610,6.323602,14.364984,-1.264001
2,high,3648,18.266582,31.230327,-7.531439
3,very_high,1252,37.735903,60.443269,-17.068490
4,extreme,586,152.879454,219.797158,-103.353857


A monotonically rising MAE across regimes means the model degrades exactly where accuracy matters most; a large positive bias in the extreme row means it systematically under-predicts hazardous air.


### Result — a systematic −103 µg/m³ bias on extremes

| Regime | Count | MAE | RMSE | **Bias** |
|---|---:|---:|---:|---:|
| normal | 206,703 | 3.29 | 9.85 | +0.80 |
| moderate | 43,610 | 6.32 | 14.36 | −1.26 |
| high | 3,648 | 18.27 | 31.23 | −7.53 |
| very_high | 1,252 | 37.74 | 60.44 | −17.07 |
| **extreme** | **586** | **152.88** | 219.80 | **−103.35** |

**The bias column is the diagnosis.** It is monotonic: near zero for ordinary air, and −103 µg/m³ for extremes. The model does not fail randomly during severe episodes — it fails *in one direction*, consistently under-predicting hazardous air by about 100 µg/m³.

That is textbook regression-to-the-mean under a squared-error objective, and it is precisely what the fixes in `MODEL_IMPROVEMENTS.md` §2.5–2.7 target: `log1p` transformation, quantile/asymmetric loss, and genuine tail weighting. A model that under-predicts hazard is worse than one that is merely imprecise — it is biased toward silence.


## 13. Spatial holdout

Hold out entire stations.

This tests:

> Can the model generalize to a station/location it has never seen?

Do not use station IDs as a shortcut feature in the production model.

In [19]:
# ============================================================================
# SPATIAL HOLDOUT - can the model generalise to a station it has never seen?
# ============================================================================
# The temporal split answers "can it forecast forward at KNOWN stations?".
# That is not the product claim. A 1-km hyper-local estimator asserts it can
# score a grid cell with no monitor in it, and only a station holdout tests
# that. Expect materially worse numbers here - that gap IS the finding.
#
# FIXED: the previous version computed the split and then never trained on it,
# so the notebook claimed a spatial holdout it never performed.
# ============================================================================
stations = np.array(sorted(df.location_id.dropna().unique()))
rng = np.random.default_rng(42)
rng.shuffle(stations)
cut_i = int(len(stations) * 0.80)
train_stations, test_stations = set(stations[:cut_i]), set(stations[cut_i:])

sp_train = df[df.location_id.isin(train_stations)].dropna(subset=[TARGET])
sp_test = df[df.location_id.isin(test_stations)].dropna(subset=[TARGET])
print(f"Spatial split: {len(train_stations)} train / {len(test_stations)} test stations")
print(f"Rows         : {len(sp_train):,} / {len(sp_test):,}")

if HAS_LGBM and len(sp_test) > 100:
    sp_imputer = SimpleImputer(strategy="median")      # local, never the shared one
    sp_model = LGBMRegressor(n_estimators=800, learning_rate=0.03, num_leaves=63,
                             random_state=42, verbose=-1)
    sp_model.fit(sp_imputer.fit_transform(sp_train[FEATURES]), sp_train[TARGET])
    sp_pred = sp_model.predict(sp_imputer.transform(sp_test[FEATURES]))

    spatial_scores = regression_metrics(sp_test[TARGET], sp_pred)

    # Like-for-like: fit the SAME architecture on the temporal split. Comparing
    # a spatial LightGBM against whatever topped the ranking (which may be the
    # persistence baseline) would compare two different things.
    tmp_imputer = SimpleImputer(strategy="median")
    tmp_model = LGBMRegressor(n_estimators=800, learning_rate=0.03, num_leaves=63,
                              random_state=42, verbose=-1)
    tmp_model.fit(tmp_imputer.fit_transform(train[FEATURES]), train[TARGET])
    temporal_scores = regression_metrics(
        y_test, tmp_model.predict(tmp_imputer.transform(test[FEATURES])))

    print("\nSame LightGBM architecture, two holdout types:")
    print("  TEMPORAL (unseen time, known stations) :", temporal_scores)
    print("  SPATIAL  (unseen stations)             :", spatial_scores)
    gap = temporal_scores["R2"] - spatial_scores["R2"]
    print(f"\nR2 change temporal -> spatial: {gap:+.3f}")
    if gap > 0.10:
        print("A large DROP means the model memorises per-station offsets rather "
              "than learning transferable physics. The honest number for a "
              "hyper-local product is the SPATIAL one.")
    else:
        print("Little or no drop: the model transfers to unseen stations about as "
              "well as to unseen time. Note the two test sets cover different "
              "periods and season mixes, so this is indicative, not a controlled "
              "comparison.")
    display(regime_metrics(sp_test[TARGET], sp_pred))
else:
    spatial_scores = None
    print("Skipped spatial model (needs lightgbm and >100 test rows).")


Spatial split: 119 train / 30 test stations
Rows         : 1,358,815 / 346,288



Same LightGBM architecture, two holdout types:
  TEMPORAL (unseen time, known stations) : {'MAE': 4.4200538634136155, 'RMSE': 14.18397733952406, 'R2': 0.5126234364555555}
  SPATIAL  (unseen stations)             : {'MAE': 5.263411078236425, 'RMSE': 17.32726343718634, 'R2': 0.6831379535910084}

R2 change temporal -> spatial: -0.171
Little or no drop: the model transfers to unseen stations about as well as to unseen time. Note the two test sets cover different periods and season mixes, so this is indicative, not a controlled comparison.


,regime,count,MAE,RMSE,bias
0,normal,191882,3.470500,8.305609,1.899466
1,moderate,111355,4.765912,9.321586,0.179501
2,high,30680,8.101855,13.607033,-2.297264
3,very_high,10494,15.337935,25.021844,-9.194532
4,extreme,1877,115.343715,191.458960,-106.745749


### Result — transfers to unseen stations, with a caveat

Same LightGBM architecture, two holdout types:

| Holdout | MAE | RMSE | R² |
|---|---:|---:|---:|
| TEMPORAL (unseen time, known stations) | 4.42 | 14.18 | 0.513 |
| SPATIAL (30 unseen stations) | 5.26 | 17.33 | **0.683** |

The spatial score is *higher*, which looks surprising. **It is not evidence that spatial generalisation is easy** — the two test sets cover different time periods and season mixes. The spatial test spans all 19 months (including the dirty seasons the model handles well), while the temporal test is monsoon-only. This is indicative, not a controlled comparison.

What can be said: the model does **not** collapse on unseen stations, so it is not purely memorising per-station offsets. The extreme bias persists though — **−106.7 µg/m³** on the spatial holdout, essentially identical to the temporal one. That bias is a property of the loss function, not of the split.

For a genuinely controlled answer, hold out stations *and* fix the time window to the same period for both arms.


## 14. Multi-horizon baseline comparison

Repeat the same experiment for:

```text
1h, 3h, 6h, 12h, 24h, 48h
```

Do not assume a model that is best at 1h is best at 24h.

In [20]:
# ============================================================================
# HORIZON SWEEP - how fast does skill decay with lead time?
# ============================================================================
# The operational question: how far ahead can AeroPulse warn before its
# forecast is no better than persistence? Skill always decays with horizon;
# where it crosses the baseline is the product's usable lead time.
#
# FIXED: the previous version called imputer.fit_transform() on the SHARED
# imputer inside this loop, silently refitting the object that the model
# comparison above depends on. Each horizon now fits its own.
# ============================================================================
horizon_results = []
if HAS_LGBM:
    for h in [1, 3, 6, 12, 24, 48]:
        target_h = f"target_pm25_t_plus_{h}h"
        if target_h not in df.columns:
            continue
        subset = df.dropna(subset=[target_h]).sort_values("timestamp_utc")
        if len(subset) < 500:
            continue
        cut_h = subset.timestamp_utc.quantile(0.85)
        tr, te = subset[subset.timestamp_utc < cut_h], subset[subset.timestamp_utc >= cut_h]
        if len(te) < 100:
            continue

        imp_h = SimpleImputer(strategy="median")       # local to this horizon
        m = LGBMRegressor(n_estimators=800, learning_rate=0.03, num_leaves=63,
                          random_state=42, verbose=-1)
        m.fit(imp_h.fit_transform(tr[FEATURES]), tr[target_h])
        p = m.predict(imp_h.transform(te[FEATURES]))

        base_p = te["pm25"].to_numpy()                 # persistence at this horizon
        horizon_results.append({
            "horizon_h": h,
            **regression_metrics(te[target_h], p),
            "persistence_RMSE": root_mean_squared_error(te[target_h], base_p),
            "extreme_recall": event_metrics(te[target_h], p)["recall"],
        })

    hr = pd.DataFrame(horizon_results)
    if not hr.empty:
        hr["skill_vs_persistence"] = 1 - hr["RMSE"] / hr["persistence_RMSE"]
        display(hr)
        useful = hr[hr.skill_vs_persistence > 0]
        if len(useful):
            print(f"Model beats persistence out to {useful.horizon_h.max()}h "
                  f"(best skill {useful.skill_vs_persistence.max():.1%} at "
                  f"{useful.loc[useful.skill_vs_persistence.idxmax(),'horizon_h']:.0f}h).")
        else:
            print("WARNING: the model never beats persistence at any horizon.")
else:
    hr = pd.DataFrame()
    print("Skipped horizon sweep (lightgbm not installed).")


,horizon_h,MAE,RMSE,R2,persistence_RMSE,extreme_recall,skill_vs_persistence
0,1,4.389449,14.046903,0.521998,16.003864,0.363176,0.122281
1,3,6.407553,16.555657,0.336054,21.014449,0.067568,0.212177
2,6,7.126864,17.181141,0.285005,22.565861,0.013514,0.238622
3,12,7.545045,17.620011,0.248189,23.970134,0.005068,0.264918
4,24,7.885281,17.949138,0.220076,23.310243,0.000000,0.229989
5,48,8.566910,18.607534,0.162440,24.350978,0.000000,0.235861


Model beats persistence out to 48h (best skill 26.5% at 12h).


### Result — useful to 48 h for levels, useless past 6 h for alerts

| Horizon | MAE | RMSE | R² | Persistence RMSE | Skill | **Extreme recall** |
|---|---:|---:|---:|---:|---:|---:|
| 1 h | 4.39 | 14.05 | 0.522 | 16.00 | +12.2% | **0.363** |
| 3 h | 6.41 | 16.56 | 0.336 | 21.01 | +21.2% | 0.068 |
| 6 h | 7.13 | 17.18 | 0.285 | 22.57 | +23.9% | 0.014 |
| 12 h | 7.55 | 17.62 | 0.248 | 23.97 | **+26.5%** | 0.005 |
| 24 h | 7.89 | 17.95 | 0.220 | 23.31 | +23.0% | **0.000** |
| 48 h | 8.57 | 18.61 | 0.162 | 24.35 | +23.6% | **0.000** |

**Two different stories in one table.**

For *concentration levels*, the model beats persistence at every horizon, and its advantage **grows** with lead time (peaking at +26.5% skill at 12 h) — persistence degrades faster than the model does. That is a real, usable forecasting capability out to 48 h.

For *extreme-event alerting*, skill collapses almost immediately: recall falls from 0.36 at 1 h to 0.07 at 3 h and **exactly zero from 24 h onward**. Beyond a few hours the model never predicts a value above 150 at all.

So the honest product claim today is: **"reliable 48-hour concentration forecasts, but hazard alerting only within ~3 hours."** Extending the alerting horizon is a loss-function and target-design problem — predicting `target_extreme_within_24h` as a *classification* task (as Phase 6 does) rather than thresholding a regression output is the right direction, and Phase 6 achieves a 43-hour median lead time that way.


## 15. Persist experiment results

Save:

```text
artifacts/pm25/baseline_comparison.csv
artifacts/pm25/baseline_regime_metrics.csv
artifacts/pm25/baseline_experiment.json
```

Do not promote a model from this notebook.

Phase 5 identifies the strongest baseline and the failure modes that Phase 6 must address.

In [21]:
# ============================================================================
# PERSIST PHASE-5 ARTIFACTS
# ============================================================================
BASELINE_DIR = ARTIFACT_DIR / "phase5"
BASELINE_DIR.mkdir(parents=True, exist_ok=True)

comparison.to_csv(BASELINE_DIR / "baseline_comparison.csv", index=False)
regime = regime_metrics(y_test, best_pred)
regime.to_csv(BASELINE_DIR / "baseline_regime_metrics.csv", index=False)
if not hr.empty:
    hr.to_csv(BASELINE_DIR / "horizon_sweep.csv", index=False)
if not cv_lgbm.empty:
    cv_lgbm.to_csv(BASELINE_DIR / "rolling_origin_cv.csv", index=False)
if not season_cv.empty:
    season_cv.to_csv(BASELINE_DIR / "leave_one_season_out.csv", index=False)

experiment = {
    "dataset": str(EVENT_FILE),
    "target": TARGET,
    "horizon_hours": TARGET_H,
    "n_features": len(FEATURES),
    "features": FEATURES,
    "rows": {"train": int(len(train)), "valid": int(len(valid)), "test": int(len(test))},
    "test_window": [str(test.timestamp_utc.min()), str(test.timestamp_utc.max())],
    "best_model_by_current_ranking": best_name,
    "ranking_rule": "extreme_recall desc, then RMSE asc",
    "comparison": comparison.to_dict("records"),
    "regime_metrics": regime.to_dict("records"),
    "spatial_holdout": spatial_scores,
    "rolling_origin_cv": cv_lgbm.to_dict("records") if not cv_lgbm.empty else None,
    "rolling_cv_mean_r2": float(cv_lgbm.R2.mean()) if not cv_lgbm.empty else None,
    "rolling_cv_std_r2": float(cv_lgbm.R2.std()) if not cv_lgbm.empty else None,
    "leave_one_season_out": season_cv.to_dict("records") if not season_cv.empty else None,
    "split_strategy": "chronological 70/15/15 + rolling-origin CV (purged) + spatial holdout",
    "random_split_rejected_because": "autoregressive lag features leak across a shuffle; see the leakage demo cell",
    "extreme_rows_in_test": int((y_test >= 150).sum()),
}
with open(BASELINE_DIR / "baseline_experiment.json", "w") as f:
    json.dump(experiment, f, indent=2, default=str)

print("Saved Phase-5 artifacts to", BASELINE_DIR)
for p in sorted(BASELINE_DIR.glob("*")):
    print("  ", p.name)


Saved Phase-5 artifacts to /Users/sunil/Documents/hackathon/AeroPulse/AeroPulse/AeroPulse_ML_Notebooks/pm25_estimator/artifacts/pm25/phase5
   baseline_comparison.csv
   baseline_experiment.json
   baseline_regime_metrics.csv
   horizon_sweep.csv
   leave_one_season_out.csv
   rolling_origin_cv.csv
